# Re-run the First-Version IBD Case Study Through the New MicrobioLink Pipeline

This notebook is a reproducible scaffold for rerunning the *same* biological case study described in the iScience paper with the current MicrobioLink workflow.

It is designed around the exact steps you requested:

1. rebuild the microbial protein table from Supplementary Table 3,
2. recreate the extracellular/plasma-membrane human candidate universe,
3. download FASTA and Pfam inputs for both sides,
4. standardize the ileum and rectum cohort DEG inputs,
5. run forward DMI, reverse DMI, and DDI,
6. apply Monte Carlo filtering with AIUPred-backed structure scoring to forward and reverse DMI,
7. combine the upstream host-microbe evidence,
8. run TieDIE separately for ileum and rectum using the directed OmniPath network and the tissue-specific DEG lists.

Important implementation notes:

- The exact historical human candidate set is the hardest part to match. This notebook therefore supports two modes:
  - preferred: provide a frozen historical accession table derived from the original HPA/LocDB snapshot,
  - fallback: query the current OmniPath intercell resource for plasma-membrane and secreted proteins.
- The TieDIE section no longer asks for an average-expression matrix. It builds the pathway input from the full directed OmniPath network and derives downstream heats directly from the DEG tables.
- Reverse DMI Monte Carlo filtering is handled by reshaping the reverse-DMI table so the same AIUPred-backed motif filter can run on the microbial motif-bearing proteins.
- This notebook is intentionally not pre-executed.


In [ ]:
from __future__ import annotations

import importlib
from pathlib import Path
import os
import sys

import pandas as pd
from IPython.display import display

search_root = Path.cwd().resolve()
repo_root = None

for candidate in [search_root, *search_root.parents]:
    if (candidate / 'notebook_runs' / 'ibd_first_version_reproduction_support.py').exists():
        repo_root = candidate
        break

if repo_root is None:
    raise FileNotFoundError(
        'Could not locate the repository root from the current working directory.',
    )

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from notebook_runs import ibd_first_version_reproduction_support as support

support = importlib.reload(support)
PIPELINE_ROOT = support.choose_pipeline_root(repo_root)
MONTE_CARLO_ROOT = support.choose_monte_carlo_root(repo_root)
support.activate_pipeline_root(PIPELINE_ROOT)
run_monte_carlo_filter = support.load_run_monte_carlo_filter(MONTE_CARLO_ROOT)

from microbiolink_api import (
    bidirectional_interactions_to_dataframe,
    ddi_interactions_to_dataframe,
    interactions_to_dataframe,
    predict_domain_domain_interactions,
    predict_domain_motif_interactions,
    predict_reverse_domain_motif_interactions,
    write_bidirectional_domain_motif_interactions,
    write_domain_domain_interactions,
    write_domain_motif_interactions,
)

print(f'Repository root: {repo_root}')
print(f'Pipeline root: {PIPELINE_ROOT}')
print(f'Monte Carlo root: {MONTE_CARLO_ROOT}')
print('Support module reloaded from disk.')
print('Restart the kernel if you change either root after importing microbiolink modules.')

## Optional Environment Setup

Enable the next cell only if the current notebook kernel still needs the package itself plus the optional IDR and TieDIE extras.


In [ ]:
INSTALL_PIPELINE_EXTRAS = False

if INSTALL_PIPELINE_EXTRAS:
    support.run_command(
        [
            sys.executable,
            '-m',
            'pip',
            'install',
            '-e',
            '.[idr,workflow]',
        ],
        cwd = PIPELINE_ROOT,
    )
else:
    print(
        'Skipping installation. Set INSTALL_PIPELINE_EXTRAS = True '
        'if this environment still needs microbiolink[idr,workflow].',
    )


## Configuration

Update the placeholder paths below before running the notebook.

The default `mmc3_filtered_oma` mode expects the staged OMA outputs produced by `tools/reproduce_ibd_paper_orthology.py`, plus local copies of `mmc3` and the tissue-specific DEG tables.


In [ ]:
MICROBIAL_INPUT_MODE = 'mmc3_filtered_oma'

SUPPLEMENTARY_TABLE_3_PATH = repo_root / 'input_data' / 'mmc4.xlsx'
SUPPLEMENTARY_TABLE_3_SHEET = 0
SUPPLEMENTARY_SPECIES_COLUMN = None
SUPPLEMENTARY_OMA_GROUP_COLUMN = 'Orthology_group_id'
SUPPLEMENTARY_MICROBIAL_PROTEIN_COLUMN = None

OMA_MEMBERSHIP_TABLE_PATH = repo_root / 'runs' / 'ibd_paper_orthology' / '03_processed' / 'oma_group_members.tsv'
OMA_MEMBERSHIP_GROUP_COLUMN = 'oma_group'
OMA_MEMBERSHIP_PROTEIN_COLUMN = 'microbial_protein'
OMA_MEMBERSHIP_SPECIES_COLUMN = 'species'
OMA_MEMBERSHIP_LABEL_COLUMN = 'oma_member_label'

MMC3_PATH = repo_root / 'input_data' / 'mmc3.csv'
MMC3_SET_COLUMN = 'Set'
MMC3_FEATURE_COLUMN = 'Feature'
MMC3_EFFECT_COLUMN = 'Dysbiosis Coefficient (CD)'
MMC3_ALLOWED_SETS = getattr(
    support,
    'MMC3_DEFAULT_SET_NAMES',
    [
        'protein abundances',
        'KO-level protein abundances',
        'EC-level protein abundances',
        'KO transcription',
        'EC transcription',
        'metagenomic KO profiles',
        'Metagenomic EC profiles',
    ],
).copy()
MMC3_CD_INCREASED_ONLY = True

HUMAN_CANDIDATE_SOURCE = 'hpa_export'

HUMAN_CANDIDATE_ACCESSIONS_PATH = (
    repo_root / 'input_data' / 'historical_human_extracellular_accessions.tsv'
)
HUMAN_CANDIDATE_ACCESSION_COLUMN = 'uniprot'
HUMAN_CANDIDATE_GENE_COLUMN = 'genesymbol'

HPA_EXPORT_PATH = repo_root / 'input_data' / 'protein_class_Predicted.tsv'
HPA_EXPORT_SHEET = 0
HPA_GENE_COLUMN = 'Gene'
HPA_ACCESSION_COLUMN = 'Uniprot'
HPA_FILTER_COLUMNS = ['Subcellular location', 'Secretome location']
HPA_INCLUDE_KEYWORDS = ['plasma membrane', 'secreted']
HPA_EXCLUDE_KEYWORDS = []

OMNIPATH_LOCATION_FILTERS = support.DEFAULT_LOCATION_FILTERS.copy()

ILEUM_DEG_PATH = repo_root / 'input_data' / 'ileum_CD_DEGs.csv'
RECTUM_DEG_PATH = repo_root / 'input_data' / 'rectum_CD_DEGs.csv'
DEG_GENE_COLUMN = 'Gene'
DEG_VALUE_COLUMN = 'Log fold change (logFC)'
DEG_PVALUE_COLUMN = 'FDR Pvalue'

MC_ITERATIONS = 1000
MC_ALPHA = 0.05
MC_DISORDER_THRESHOLD = 0.60
MC_BINDING_THRESHOLD = 0.60
MC_MIN_SUPPORT_FRACTION = 1.0
MC_REQUIRE_BINDING = True
MC_SEED = 0
MC_FORCE_CPU = True
MC_GPU = 0
# Use a kernel where `iupred` and `torch` are available so
# AIUPred-backed Monte Carlo filtering can run in this notebook.
# Set to 'auto' to allow fallback, True to require Monte Carlo, or False to skip it.
RUN_MONTE_CARLO = True
# Optional precomputed residue-level score tables.
# Expected columns include: human_protein, position, disorder_score, binding_score.
# For reverse DMI, keep the protein-id column name as `human_protein` because the same
# Monte Carlo reader is reused on the microbial motif-bearing proteins.
FORWARD_STRUCTURE_SCORES_PATH = None
REVERSE_STRUCTURE_SCORES_PATH = None

DEDUPLICATE_TIEDIE_PAIRS = True
RUN_TIEDIE = True
TIEDIE_EXECUTABLE = 'tiedie'
TIEDIE_PERMUTE = 1000

OUTPUT_ROOT = repo_root / 'notebook_runs' / 'ibd_same_input_new_microbiolink_run'
MMC3_UNIPROT_ANNOTATION_PATH = OUTPUT_ROOT / '00_inputs' / 'member_uniprot_annotations.tsv'
MMC3_KEGG_KO_MAPPING_PATH = OUTPUT_ROOT / '00_inputs' / 'member_kegg_ko.tsv'
MMC3_DOWNLOAD_UNIPROT_ANNOTATIONS = True
MMC3_DOWNLOAD_KEGG_KO_MAPPING = True
MICROBIAL_FASTA_SOURCE_DIR = repo_root / 'runs' / 'ibd_paper_orthology' / '02_oma_run' / 'DB'


In [ ]:
required_input_paths = [
    ILEUM_DEG_PATH,
    RECTUM_DEG_PATH,
]

if MICROBIAL_INPUT_MODE == 'supplementary_table':
    required_input_paths.append(SUPPLEMENTARY_TABLE_3_PATH)
elif MICROBIAL_INPUT_MODE == 'mmc3_filtered_oma':
    required_input_paths.extend(
        [
            MMC3_PATH,
            OMA_MEMBERSHIP_TABLE_PATH,
        ],
    )
else:
    raise ValueError(
        'MICROBIAL_INPUT_MODE must be either supplementary_table or mmc3_filtered_oma.',
    )

if HUMAN_CANDIDATE_SOURCE == 'generic_accession_table':
    required_input_paths.append(HUMAN_CANDIDATE_ACCESSIONS_PATH)
elif HUMAN_CANDIDATE_SOURCE == 'hpa_export':
    required_input_paths.append(HPA_EXPORT_PATH)
elif HUMAN_CANDIDATE_SOURCE == 'current_omnipath_intercell_fallback':
    pass
else:
    raise ValueError(
        'HUMAN_CANDIDATE_SOURCE must be one of: '
        'generic_accession_table, hpa_export, current_omnipath_intercell_fallback.'
    )

if (
    MICROBIAL_INPUT_MODE == 'supplementary_table'
    and OMA_MEMBERSHIP_TABLE_PATH is not None
    and SUPPLEMENTARY_MICROBIAL_PROTEIN_COLUMN is None
):
    required_input_paths.append(Path(OMA_MEMBERSHIP_TABLE_PATH))

optional_structure_score_paths = {
    'forward': FORWARD_STRUCTURE_SCORES_PATH,
    'reverse': REVERSE_STRUCTURE_SCORES_PATH,
}

required_input_paths.extend(
    Path(path)
    for path in optional_structure_score_paths.values()
    if path is not None
)

missing_paths = [
    Path(path)
    for path in required_input_paths
    if not Path(path).exists()
]

if missing_paths:
    raise FileNotFoundError(
        'Update the configuration cell before running the notebook. '
        'Missing paths:\n' + '\n'.join(
            str(path)
            for path in missing_paths
        ),
    )

directories = {
    'inputs': support.ensure_directory(OUTPUT_ROOT / '00_inputs'),
    'microbial': support.ensure_directory(OUTPUT_ROOT / '01_microbial'),
    'human': support.ensure_directory(OUTPUT_ROOT / '02_human'),
    'deg': support.ensure_directory(OUTPUT_ROOT / '03_deg'),
    'interactions': support.ensure_directory(OUTPUT_ROOT / '04_interactions'),
    'tiedie': support.ensure_directory(OUTPUT_ROOT / '05_tiedie'),
}

aiupred_runtime = support.inspect_aiupred_runtime()
missing_structure_score_sides = [
    side
    for side, path in optional_structure_score_paths.items()
    if path is None
]

if RUN_MONTE_CARLO not in ['auto', True, False]:
    raise ValueError(
        'RUN_MONTE_CARLO must be one of: auto, True, False.',
    )

monte_carlo_ready = (
    not missing_structure_score_sides
    or aiupred_runtime['aiupred_ready']
)

if RUN_MONTE_CARLO is True and not monte_carlo_ready:
    raise ImportError(
        'Monte Carlo filtering needs either a working AIUPred runtime '
        '(the `iupred` package plus PyTorch) or precomputed structure '
        'score tables for every missing side. Missing structure-score '
        f'inputs: {missing_structure_score_sides}. Current AIUPred '
        f"status: {aiupred_runtime['message']} Set "
        'FORWARD_STRUCTURE_SCORES_PATH and/or '
        'REVERSE_STRUCTURE_SCORES_PATH in the configuration cell, or '
        'install PyTorch so AIUPred can run locally.'
    )

RUN_MONTE_CARLO_EFFECTIVE = (
    monte_carlo_ready
    if RUN_MONTE_CARLO == 'auto'
    else bool(RUN_MONTE_CARLO)
)
monte_carlo_status = (
    'enabled'
    if RUN_MONTE_CARLO_EFFECTIVE
    else 'skipped'
)
monte_carlo_status_reason = (
    'Monte Carlo will run with AIUPred or supplied structure-score tables.'
    if RUN_MONTE_CARLO_EFFECTIVE
    else 'Monte Carlo is disabled for this notebook run. Raw DMI and reverse-DMI '
    'tables will be passed downstream without motif-level filtering.'
)

display(
    pd.DataFrame(
        {
            'logical_name': list(directories.keys()),
            'path': [str(path) for path in directories.values()],
        },
    ),
)

display(
    pd.DataFrame(
        [
            aiupred_runtime | {
                'run_monte_carlo': str(RUN_MONTE_CARLO),
                'run_monte_carlo_effective': RUN_MONTE_CARLO_EFFECTIVE,
                'monte_carlo_status': monte_carlo_status,
                'monte_carlo_status_reason': monte_carlo_status_reason,
                'forward_structure_scores_path': str(FORWARD_STRUCTURE_SCORES_PATH),
                'reverse_structure_scores_path': str(REVERSE_STRUCTURE_SCORES_PATH),
            },
        ],
    ),
)

## Step 1: Prepare the Microbial Protein List

This step creates the exact three-column table used by the rerun:

- `species`
- `oma_group`
- `microbial_protein`

For the current local setup, the default route is `mmc3_filtered_oma`, because the fresh local OMA group identifiers are not guaranteed to match the paper's original `mmc4` orthogroup IDs.

The alternative `supplementary_table` route is still available, but it should only be used when the orthogroup identifiers in Supplementary Table 3 are aligned to the membership table you provide.

In [ ]:
mmc3_filter_result = None

if MICROBIAL_INPUT_MODE == 'supplementary_table':
    microbial_proteins = support.prepare_microbial_protein_table(
        supplementary_table_path = SUPPLEMENTARY_TABLE_3_PATH,
        sheet_name = SUPPLEMENTARY_TABLE_3_SHEET,
        species_column = SUPPLEMENTARY_SPECIES_COLUMN,
        oma_group_column = SUPPLEMENTARY_OMA_GROUP_COLUMN,
        microbial_protein_column = SUPPLEMENTARY_MICROBIAL_PROTEIN_COLUMN,
        membership_table_path = OMA_MEMBERSHIP_TABLE_PATH,
        membership_group_column = OMA_MEMBERSHIP_GROUP_COLUMN,
        membership_protein_column = OMA_MEMBERSHIP_PROTEIN_COLUMN,
        membership_species_column = OMA_MEMBERSHIP_SPECIES_COLUMN,
    )
elif MICROBIAL_INPUT_MODE == 'mmc3_filtered_oma':
    mmc3_filter_result = support.prepare_microbial_protein_table_from_mmc3(
        membership_table_path = OMA_MEMBERSHIP_TABLE_PATH,
        mmc3_path = MMC3_PATH,
        membership_group_column = OMA_MEMBERSHIP_GROUP_COLUMN,
        membership_protein_column = OMA_MEMBERSHIP_PROTEIN_COLUMN,
        membership_species_column = OMA_MEMBERSHIP_SPECIES_COLUMN,
        membership_label_column = OMA_MEMBERSHIP_LABEL_COLUMN,
        mmc3_set_column = MMC3_SET_COLUMN,
        mmc3_feature_column = MMC3_FEATURE_COLUMN,
        mmc3_effect_column = MMC3_EFFECT_COLUMN,
        mmc3_allowed_sets = MMC3_ALLOWED_SETS,
        cd_increased_only = MMC3_CD_INCREASED_ONLY,
        uniprot_annotation_path = MMC3_UNIPROT_ANNOTATION_PATH,
        kegg_ko_mapping_path = MMC3_KEGG_KO_MAPPING_PATH,
        download_uniprot_annotations = MMC3_DOWNLOAD_UNIPROT_ANNOTATIONS,
        download_kegg_ko_mapping = MMC3_DOWNLOAD_KEGG_KO_MAPPING,
    )
    microbial_proteins = mmc3_filter_result.microbial_proteins
else:
    raise ValueError(
        'MICROBIAL_INPUT_MODE must be either supplementary_table or mmc3_filtered_oma.',
    )

microbial_table_path = directories['microbial'] / 'microbial_protein_table.tsv'
microbial_accessions_path = directories['microbial'] / 'microbial_accessions.tsv'

microbial_proteins.to_csv(microbial_table_path, sep = '	', index = False)
support.write_identifier_table(
    microbial_proteins['microbial_protein'].tolist(),
    microbial_accessions_path,
    column_name = 'uniprot',
)

summary_rows = [
    {
        'metric': 'microbial_rows',
        'value': len(microbial_proteins),
    },
    {
        'metric': 'microbial_unique_groups',
        'value': microbial_proteins['oma_group'].nunique(),
    },
    {
        'metric': 'microbial_unique_proteins',
        'value': microbial_proteins['microbial_protein'].nunique(),
    },
    {
        'metric': 'microbial_output_file',
        'value': str(microbial_table_path),
    },
]

if mmc3_filter_result is not None:
    mmc3_annotation_output_path = directories['inputs'] / 'mmc3_member_annotations.tsv'
    mmc3_feature_token_output_path = directories['inputs'] / 'mmc3_feature_tokens.tsv'
    mmc3_match_evidence_output_path = directories['inputs'] / 'mmc3_match_evidence.tsv'
    mmc3_matched_groups_output_path = directories['inputs'] / 'mmc3_matched_groups.tsv'

    mmc3_filter_result.member_annotations.to_csv(
        mmc3_annotation_output_path,
        sep = '	',
        index = False,
    )
    mmc3_filter_result.mmc3_feature_tokens.to_csv(
        mmc3_feature_token_output_path,
        sep = '	',
        index = False,
    )
    mmc3_filter_result.match_evidence.to_csv(
        mmc3_match_evidence_output_path,
        sep = '	',
        index = False,
    )
    mmc3_filter_result.matched_groups.to_csv(
        mmc3_matched_groups_output_path,
        sep = '	',
        index = False,
    )

    summary_rows.extend(
        [
            {
                'metric': 'mmc3_match_rows',
                'value': len(mmc3_filter_result.match_evidence),
            },
            {
                'metric': 'mmc3_matched_groups',
                'value': len(mmc3_filter_result.matched_groups),
            },
            {
                'metric': 'mmc3_uniprot_annotation_cache',
                'value': str(MMC3_UNIPROT_ANNOTATION_PATH),
            },
            {
                'metric': 'mmc3_kegg_ko_cache',
                'value': str(MMC3_KEGG_KO_MAPPING_PATH),
            },
        ],
    )

display(pd.DataFrame(summary_rows))
display(microbial_proteins.head(20))

if mmc3_filter_result is not None:
    display(mmc3_filter_result.matched_groups.head(20))

## Step 2: Prepare Microbial FASTA and Pfam Annotations

For the current local rerun, microbial protein sequences are extracted from the local OMA input proteomes rather than re-downloaded from UniProt.

Pfam annotations are still retrieved from UniProt, but only for the subset of microbial accessions that are valid UniProt identifiers. Proteins that only exist as local assembly accessions, such as the `Ruminococcus_gnavus` `CUN...` and `CUO...` records, are retained in the microbial FASTA for reverse DMI but will not contribute to the forward DMI or DDI domain-based steps unless an external Pfam annotation source is added later.

In [ ]:
bacterial_fasta_path = directories['microbial'] / 'microbial_proteins.fasta'
bacterial_raw_domain_path = directories['microbial'] / 'microbial_uniprot_annotations.tsv'
bacterial_domain_path = directories['microbial'] / 'microbial_domains.tsv'
bacterial_domain_input_path = directories['microbial'] / 'microbial_domain_query_accessions.tsv'
bacterial_domain_missing_path = directories['microbial'] / 'microbial_domain_missing_accessions.tsv'

microbial_uniprot_accessions, microbial_non_uniprot_accessions = support.partition_accessions_by_uniprot_pattern(
    microbial_proteins['microbial_protein'].tolist(),
)
reference_fasta_paths = sorted(Path(MICROBIAL_FASTA_SOURCE_DIR).glob('*.fa'))
_, missing_fasta_accessions = support.write_selected_fasta_from_reference_fastas(
    microbial_proteins['microbial_protein'].tolist(),
    source_fasta_paths = reference_fasta_paths,
    output_path = bacterial_fasta_path,
    normalize_headers = True,
)

support.write_identifier_table(
    microbial_uniprot_accessions,
    bacterial_domain_input_path,
    column_name = 'microbial_protein',
)

if microbial_uniprot_accessions:
    bacterial_domains = support.download_uniprot_domain_table(
        microbial_uniprot_accessions,
        raw_output_path = bacterial_raw_domain_path,
        domain_output_path = bacterial_domain_path,
    )
else:
    bacterial_domains = pd.DataFrame(columns = ['protein', 'pfam'])
    bacterial_domains.to_csv(bacterial_domain_path, sep = '	', index = False)
    pd.DataFrame().to_csv(bacterial_raw_domain_path, sep = '	', index = False)

bacterial_domain_missing = microbial_proteins.loc[
    ~microbial_proteins['microbial_protein'].isin(bacterial_domains['protein']),
    ['species', 'oma_group', 'microbial_protein'],
].drop_duplicates()
bacterial_domain_missing.to_csv(
    bacterial_domain_missing_path,
    sep = '	',
    index = False,
)

display(
    pd.DataFrame(
        [
            {
                'metric': 'microbial_accessions',
                'value': microbial_proteins['microbial_protein'].nunique(),
            },
            {
                'metric': 'microbial_uniprot_like_accessions',
                'value': len(microbial_uniprot_accessions),
            },
            {
                'metric': 'microbial_non_uniprot_accessions',
                'value': len(microbial_non_uniprot_accessions),
            },
            {
                'metric': 'microbial_fasta_missing_accessions',
                'value': len(missing_fasta_accessions),
            },
            {
                'metric': 'microbial_domain_annotated_proteins',
                'value': bacterial_domains['protein'].nunique(),
            },
            {
                'metric': 'microbial_domain_missing_proteins',
                'value': bacterial_domain_missing['microbial_protein'].nunique(),
            },
            {
                'metric': 'microbial_unique_pfams',
                'value': bacterial_domains['pfam'].str.split(';').explode().nunique()
                if not bacterial_domains.empty
                else 0,
            },
        ],
    ),
)
display(bacterial_domains.head())
display(bacterial_domain_missing.head())

## Step 3: Recreate the Human Candidate Universe

This notebook now supports three human-input modes:

- `generic_accession_table`: a prebuilt two-column table of accessions and optional gene symbols,
- `hpa_export`: a wide Human Protein Atlas export with columns such as `Gene` and `Uniprot`,
- `current_omnipath_intercell_fallback`: a live fallback query against the current OmniPath intercell resource.

For your HPA file format, set `HUMAN_CANDIDATE_SOURCE = 'hpa_export'` and point `HPA_EXPORT_PATH` to the file. If the file is already the filtered candidate set, you can leave the HPA keyword filters empty.


In [ ]:
if HUMAN_CANDIDATE_SOURCE == 'generic_accession_table':
    human_candidates = support.load_accession_table(
        HUMAN_CANDIDATE_ACCESSIONS_PATH,
        accession_column = HUMAN_CANDIDATE_ACCESSION_COLUMN,
        gene_column = HUMAN_CANDIDATE_GENE_COLUMN,
    )
    human_candidate_mode = 'generic_accession_table'
elif HUMAN_CANDIDATE_SOURCE == 'hpa_export':
    human_candidates = support.load_hpa_candidate_accession_table(
        HPA_EXPORT_PATH,
        accession_column = HPA_ACCESSION_COLUMN,
        gene_column = HPA_GENE_COLUMN,
        sheet_name = HPA_EXPORT_SHEET,
        filter_columns = HPA_FILTER_COLUMNS,
        include_keywords = HPA_INCLUDE_KEYWORDS,
        exclude_keywords = HPA_EXCLUDE_KEYWORDS,
    )
    human_candidate_mode = 'hpa_export'
elif HUMAN_CANDIDATE_SOURCE == 'current_omnipath_intercell_fallback':
    human_candidates = support.query_current_human_candidate_accessions(
        location_filters = OMNIPATH_LOCATION_FILTERS,
    )
    human_candidate_mode = 'current_omnipath_intercell_fallback'
else:
    raise ValueError(
        'HUMAN_CANDIDATE_SOURCE must be one of: '
        'generic_accession_table, hpa_export, current_omnipath_intercell_fallback.'
    )

human_candidate_table_path = directories['human'] / 'human_candidate_accessions.tsv'
human_fasta_path = directories['human'] / 'human_extracellular.fasta'
human_raw_domain_path = directories['human'] / 'human_uniprot_annotations.tsv'
human_domain_path = directories['human'] / 'human_domains.tsv'

human_candidates.to_csv(human_candidate_table_path, sep = '\t', index = False)
support.download_uniprot_fasta(
    human_candidates['uniprot'].tolist(),
    human_fasta_path,
)
human_domains = support.download_uniprot_domain_table(
    human_candidates['uniprot'].tolist(),
    raw_output_path = human_raw_domain_path,
    domain_output_path = human_domain_path,
)

display(
    pd.DataFrame(
        [
            {
                'metric': 'human_candidate_mode',
                'value': human_candidate_mode,
            },
            {
                'metric': 'human_candidate_accessions',
                'value': human_candidates['uniprot'].nunique(),
            },
            {
                'metric': 'human_domain_annotated_proteins',
                'value': human_domains['protein'].nunique(),
            },
            {
                'metric': 'human_unique_pfams',
                'value': human_domains['pfam'].str.split(';').explode().nunique(),
            },
        ],
    ),
)
display(human_candidates.head())
display(human_domains.head())


## Step 4: Standardize the Ileum and Rectum DEG Tables

These outputs are written in a simple three-column format:

- `gene_symbol`
- `log2FC`
- `padj`

The TieDIE section below uses these standardized DEG tables directly. No average-expression matrix is required.


In [ ]:
ileum_deg = support.standardize_deg_table(
    ILEUM_DEG_PATH,
    gene_column = DEG_GENE_COLUMN,
    value_column = DEG_VALUE_COLUMN,
    pvalue_column = DEG_PVALUE_COLUMN,
)
rectum_deg = support.standardize_deg_table(
    RECTUM_DEG_PATH,
    gene_column = DEG_GENE_COLUMN,
    value_column = DEG_VALUE_COLUMN,
    pvalue_column = DEG_PVALUE_COLUMN,
)

ileum_deg_standardized_path = directories['deg'] / 'ileum_deg_standardized.tsv'
rectum_deg_standardized_path = directories['deg'] / 'rectum_deg_standardized.tsv'

ileum_deg.to_csv(ileum_deg_standardized_path, sep = '	', index = False)
rectum_deg.to_csv(rectum_deg_standardized_path, sep = '	', index = False)

display(
    pd.DataFrame(
        [
            {
                'tissue': 'ileum',
                'deg_rows': len(ileum_deg),
                'output_file': str(ileum_deg_standardized_path),
            },
            {
                'tissue': 'rectum',
                'deg_rows': len(rectum_deg),
                'output_file': str(rectum_deg_standardized_path),
            },
        ],
    ),
)


## Step 5: Run Forward DMI and Monte Carlo Filtering

This uses the human extracellular FASTA as the motif-bearing side and the microbial Pfam table as the domain-bearing side.


In [ ]:
forward_interactions = predict_domain_motif_interactions(
    fasta_file = human_fasta_path,
    bacterial_domain_file = bacterial_domain_path,
)
forward_frame = interactions_to_dataframe(forward_interactions)
forward_dmi_path = directories['interactions'] / 'forward_dmi.tsv'

write_domain_motif_interactions(
    forward_interactions,
    forward_dmi_path,
    separator = '	',
)

forward_monte_carlo_full_path = directories['interactions'] / 'forward_dmi_monte_carlo.tsv'
forward_monte_carlo_kept_path = directories['interactions'] / 'forward_dmi_monte_carlo_kept.tsv'

if RUN_MONTE_CARLO_EFFECTIVE:
    forward_monte_carlo = run_monte_carlo_filter(
        interaction_file = forward_dmi_path,
        output_file = forward_monte_carlo_full_path,
        filtered_output_file = forward_monte_carlo_kept_path,
        structure_scores = FORWARD_STRUCTURE_SCORES_PATH,
        fasta_file = human_fasta_path,
        coordinate_system = 'zero_based_half_open',
        iterations = MC_ITERATIONS,
        alpha = MC_ALPHA,
        disorder_threshold = MC_DISORDER_THRESHOLD,
        binding_threshold = MC_BINDING_THRESHOLD,
        min_support_fraction = MC_MIN_SUPPORT_FRACTION,
        require_binding = MC_REQUIRE_BINDING,
        seed = MC_SEED,
        force_cpu = MC_FORCE_CPU,
        gpu = MC_GPU,
    )
    forward_monte_carlo_kept = forward_monte_carlo.loc[
        forward_monte_carlo['passes_monte_carlo']
    ].copy()
    forward_selected_interactions = forward_monte_carlo_kept.copy()
    forward_selected_interactions_path = forward_monte_carlo_kept_path
    forward_evidence_label = 'forward_dmi_monte_carlo'
else:
    forward_monte_carlo = pd.DataFrame()
    forward_monte_carlo_kept = forward_frame.copy()
    forward_selected_interactions = forward_frame.copy()
    forward_selected_interactions_path = directories['interactions'] / 'forward_dmi_unfiltered.tsv'
    forward_selected_interactions.to_csv(
        forward_selected_interactions_path,
        sep = '	',
        index = False,
    )
    forward_evidence_label = 'forward_dmi_unfiltered'

display(
    pd.DataFrame(
        [
            {
                'metric': 'forward_dmi_raw_rows',
                'value': len(forward_frame),
            },
            {
                'metric': 'forward_selected_rows',
                'value': len(forward_selected_interactions),
            },
            {
                'metric': 'forward_dmi_unique_host_proteins',
                'value': forward_frame['human_protein'].nunique(),
            },
            {
                'metric': 'forward_monte_carlo_applied',
                'value': RUN_MONTE_CARLO_EFFECTIVE,
            },
        ],
    ),
)
display(forward_selected_interactions.head())


## Step 6: Run Reverse DMI and Monte Carlo Filtering

Reverse DMI uses the microbial FASTA as the motif-bearing side and the human Pfam table as the domain-bearing side.

For the structural filter, the reverse-DMI output is reshaped so the same Monte Carlo routine can score motifs on the microbial proteins.


In [ ]:
reverse_interactions = predict_reverse_domain_motif_interactions(
    bacterial_fasta_file = bacterial_fasta_path,
    human_domain_file = human_domain_path,
)
reverse_frame = bidirectional_interactions_to_dataframe(reverse_interactions)
reverse_dmi_path = directories['interactions'] / 'reverse_dmi.tsv'

write_bidirectional_domain_motif_interactions(
    reverse_interactions,
    reverse_dmi_path,
    separator = '	',
)

reverse_monte_carlo_input = support.build_reverse_monte_carlo_input_table(
    reverse_frame,
)
reverse_monte_carlo_input_path = directories['interactions'] / 'reverse_dmi_monte_carlo_input.tsv'
reverse_monte_carlo_input.to_csv(
    reverse_monte_carlo_input_path,
    sep = '	',
    index = False,
)

reverse_monte_carlo_full_path = directories['interactions'] / 'reverse_dmi_monte_carlo.tsv'
reverse_monte_carlo_kept_path = directories['interactions'] / 'reverse_dmi_monte_carlo_kept.tsv'

reverse_monte_carlo_annotated_path = directories['interactions'] / 'reverse_dmi_monte_carlo_annotated.tsv'

if RUN_MONTE_CARLO_EFFECTIVE:
    reverse_monte_carlo = run_monte_carlo_filter(
        interaction_file = reverse_monte_carlo_input_path,
        output_file = reverse_monte_carlo_full_path,
        filtered_output_file = reverse_monte_carlo_kept_path,
        structure_scores = REVERSE_STRUCTURE_SCORES_PATH,
        fasta_file = bacterial_fasta_path,
        coordinate_system = 'zero_based_half_open',
        iterations = MC_ITERATIONS,
        alpha = MC_ALPHA,
        disorder_threshold = MC_DISORDER_THRESHOLD,
        binding_threshold = MC_BINDING_THRESHOLD,
        min_support_fraction = MC_MIN_SUPPORT_FRACTION,
        require_binding = MC_REQUIRE_BINDING,
        seed = MC_SEED,
        force_cpu = MC_FORCE_CPU,
        gpu = MC_GPU,
    )

    reverse_monte_carlo_annotated = support.merge_reverse_monte_carlo_results(
        reverse_frame,
        reverse_monte_carlo,
    )
    reverse_monte_carlo_annotated.to_csv(
        reverse_monte_carlo_annotated_path,
        sep = '	',
        index = False,
    )

    reverse_monte_carlo_kept = reverse_monte_carlo_annotated.loc[
        reverse_monte_carlo_annotated['passes_monte_carlo'].fillna(False)
    ].copy()
    reverse_monte_carlo_kept.to_csv(
        reverse_monte_carlo_kept_path,
        sep = '	',
        index = False,
    )
    reverse_selected_interactions = reverse_monte_carlo_kept.copy()
    reverse_selected_interactions_path = reverse_monte_carlo_kept_path
    reverse_evidence_label = 'reverse_dmi_monte_carlo'
else:
    reverse_monte_carlo = pd.DataFrame()
    reverse_monte_carlo_annotated = reverse_frame.copy()
    reverse_monte_carlo_annotated.to_csv(
        reverse_monte_carlo_annotated_path,
        sep = '	',
        index = False,
    )
    reverse_monte_carlo_kept = reverse_frame.copy()
    reverse_selected_interactions = reverse_frame.copy()
    reverse_selected_interactions_path = directories['interactions'] / 'reverse_dmi_unfiltered.tsv'
    reverse_selected_interactions.to_csv(
        reverse_selected_interactions_path,
        sep = '	',
        index = False,
    )
    reverse_evidence_label = 'reverse_dmi_unfiltered'

display(
    pd.DataFrame(
        [
            {
                'metric': 'reverse_dmi_raw_rows',
                'value': len(reverse_frame),
            },
            {
                'metric': 'reverse_selected_rows',
                'value': len(reverse_selected_interactions),
            },
            {
                'metric': 'reverse_dmi_unique_microbial_proteins',
                'value': reverse_frame['microbial_protein'].nunique(),
            },
            {
                'metric': 'reverse_monte_carlo_applied',
                'value': RUN_MONTE_CARLO_EFFECTIVE,
            },
        ],
    ),
)
display(reverse_selected_interactions.head())


## Step 7: Run DDI

DDI uses the microbial and human Pfam tables directly and does not use Monte Carlo or AIUPred because there is no motif coordinate to resample.


In [ ]:
ddi_interactions = predict_domain_domain_interactions(
    bacterial_domain_file = bacterial_domain_path,
    human_domain_file = human_domain_path,
)
ddi_frame = ddi_interactions_to_dataframe(ddi_interactions)
ddi_path = directories['interactions'] / 'ddi.tsv'

write_domain_domain_interactions(
    ddi_interactions,
    ddi_path,
    separator = '	',
)

display(
    pd.DataFrame(
        [
            {
                'metric': 'ddi_rows',
                'value': len(ddi_frame),
            },
            {
                'metric': 'ddi_unique_host_proteins',
                'value': ddi_frame['human_protein'].nunique(),
            },
            {
                'metric': 'ddi_unique_microbial_proteins',
                'value': ddi_frame['bacterial_protein'].nunique(),
            },
        ],
    ),
)
display(ddi_frame.head())


## Step 8: Combine the Upstream Evidence for TieDIE

By default this notebook collapses repeated host-microbe pairs across evidence types so the TieDIE seed file does not triple-count the same pair simply because it appeared in forward DMI, reverse DMI, and DDI.


In [ ]:
forward_tiedie = support.normalize_forward_interactions_for_tiedie(
    forward_selected_interactions,
    evidence_label = forward_evidence_label,
)
reverse_tiedie = support.normalize_reverse_interactions_for_tiedie(
    reverse_selected_interactions,
    evidence_label = reverse_evidence_label,
)
ddi_tiedie = support.normalize_ddi_interactions_for_tiedie(
    ddi_frame,
    evidence_label = 'ddi',
)

combined_upstream = support.combine_tiedie_upstream_tables(
    [forward_tiedie, reverse_tiedie, ddi_tiedie],
    deduplicate_pairs = DEDUPLICATE_TIEDIE_PAIRS,
)
combined_upstream_path = directories['interactions'] / 'combined_upstream_pairs.tsv'
combined_upstream.to_csv(combined_upstream_path, sep = '\t', index = False)
combined_upstream_tiedie_path = directories['interactions'] / 'combined_upstream_pairs_tiedie.tsv'
support.write_tiedie_hmi_table(
    combined_upstream,
    combined_upstream_tiedie_path,
)

display(
    pd.DataFrame(
        [
            {
                'evidence_type': forward_evidence_label,
                'rows': len(forward_tiedie),
            },
            {
                'evidence_type': reverse_evidence_label,
                'rows': len(reverse_tiedie),
            },
            {
                'evidence_type': 'ddi',
                'rows': len(ddi_tiedie),
            },
            {
                'evidence_type': 'combined_for_tiedie',
                'rows': len(combined_upstream),
            },
        ],
    ),
)
display(combined_upstream.head())


## Step 9: Run TieDIE Separately for Ileum and Rectum

This notebook uses the full directed OmniPath network for the TieDIE pathway input and the tissue-specific DEG lists for downstream heat generation.

Inputs for this step are therefore:

- the combined host-microbe interaction table,
- the ileum or rectum DEG table,
- the live OmniPath and CollecTRI resources.

No transcriptomics average-expression matrix is required.


In [ ]:
tissue_configs = {
    'ileum': {
        'endpoint_file': ileum_deg_standardized_path,
    },
    'rectum': {
        'endpoint_file': rectum_deg_standardized_path,
    },
}

command_env = os.environ.copy()
existing_pythonpath = command_env.get('PYTHONPATH', '')
command_env['PYTHONPATH'] = (
    str(PIPELINE_ROOT)
    if not existing_pythonpath
    else str(PIPELINE_ROOT) + os.pathsep + existing_pythonpath
)

tiedie_outputs = {}

if RUN_TIEDIE:
    for tissue_name, tissue_config in tissue_configs.items():
        tissue_dir = support.ensure_directory(directories['tiedie'] / tissue_name)
        tiedie_run_dir = support.ensure_directory(tissue_dir / 'tiedie_run')

        tiedie_input_paths = support.build_tiedie_inputs_from_omnipath_network(
            endpoint_file = tissue_config['endpoint_file'],
            hmi_prediction_file = combined_upstream_tiedie_path,
            output_dir = tissue_dir,
            repo_root = repo_root,
            endpoint_separator = '\t',
            endpoint_pvalue_column = 3,
            endpoint_value_column = 2,
        )

        support.run_command(
            [
                TIEDIE_EXECUTABLE,
                '--network',
                str(tiedie_input_paths['pathway']),
                '--up_heats',
                str(tiedie_input_paths['upstream']),
                '--down_heats',
                str(tiedie_input_paths['downstream']),
                '--permute',
                str(TIEDIE_PERMUTE),
                '--output_folder',
                str(tiedie_run_dir),
            ],
            cwd = PIPELINE_ROOT,
            env = command_env,
        )

        support.run_command(
            [
                sys.executable,
                '-m',
                'microbiolink.processing_tiedie_output',
                '--tiedie_file',
                str(tiedie_run_dir / 'tiedie.cn.sif'),
                '--heats_file',
                str(tiedie_run_dir / 'heats.NA'),
                '--hmi_file',
                str(combined_upstream_tiedie_path),
                '--tf_tg_file',
                str(tiedie_input_paths['contextualised_tf_tg']),
                '--endpoint_file',
                str(tissue_config['endpoint_file']),
                '--endpoint_pvalue_column',
                '3',
                '--endpoint_value_column',
                '2',
                '--sep_endpoint',
                '\t',
                '--network_output',
                str(tissue_dir / 'final_network.tsv'),
                '--node_attr_output',
                str(tissue_dir / 'final_nodes.tsv'),
            ],
            cwd = PIPELINE_ROOT,
            env = command_env,
        )

        tiedie_outputs[tissue_name] = {
            'input_dir': tissue_dir,
            'run_dir': tiedie_run_dir,
            'pathway_input': tiedie_input_paths['pathway'],
            'upstream_input': tiedie_input_paths['upstream'],
            'downstream_input': tiedie_input_paths['downstream'],
            'tf_tg_file': tiedie_input_paths['contextualised_tf_tg'],
            'network_output': tissue_dir / 'final_network.tsv',
            'node_output': tissue_dir / 'final_nodes.tsv',
        }
else:
    print('Skipping TieDIE. Set RUN_TIEDIE = True to build both tissue-specific runs.')

display(
    pd.DataFrame(
        [
            {
                'tissue': tissue_name,
                'input_dir': str(paths['input_dir']),
                'run_dir': str(paths['run_dir']),
                'pathway_input': str(paths['pathway_input']),
                'upstream_input': str(paths['upstream_input']),
                'downstream_input': str(paths['downstream_input']),
                'final_network': str(paths['network_output']),
                'final_nodes': str(paths['node_output']),
            }
            for tissue_name, paths in tiedie_outputs.items()
        ],
    ),
)


## Step 10: Output Inventory

This last cell collects the main generated artifacts so you can jump directly to the tables and networks produced by the rerun.


In [ ]:
output_inventory = pd.DataFrame(
    [
        {
            'artifact': 'microbial_protein_table',
            'path': str(microbial_table_path),
        },
        {
            'artifact': 'microbial_fasta',
            'path': str(bacterial_fasta_path),
        },
        {
            'artifact': 'microbial_domains',
            'path': str(bacterial_domain_path),
        },
        {
            'artifact': 'microbial_domain_missing_accessions',
            'path': str(bacterial_domain_missing_path),
        },
        {
            'artifact': 'human_candidate_table',
            'path': str(human_candidate_table_path),
        },
        {
            'artifact': 'human_extracellular_fasta',
            'path': str(human_fasta_path),
        },
        {
            'artifact': 'human_domains',
            'path': str(human_domain_path),
        },
        {
            'artifact': 'ileum_deg_standardized',
            'path': str(ileum_deg_standardized_path),
        },
        {
            'artifact': 'rectum_deg_standardized',
            'path': str(rectum_deg_standardized_path),
        },
        {
            'artifact': 'forward_dmi_selected_for_downstream',
            'path': str(forward_selected_interactions_path),
        },
        {
            'artifact': 'reverse_dmi_selected_for_downstream',
            'path': str(reverse_selected_interactions_path),
        },
        {
            'artifact': 'ddi',
            'path': str(ddi_path),
        },
        {
            'artifact': 'combined_upstream_pairs',
            'path': str(combined_upstream_path),
        },
        {
            'artifact': 'combined_upstream_pairs_tiedie',
            'path': str(combined_upstream_tiedie_path),
        },
    ]
    + [
        {
            'artifact': f'{tissue_name}_final_network',
            'path': str(paths['network_output']),
        }
        for tissue_name, paths in tiedie_outputs.items()
    ]
    + [
        {
            'artifact': f'{tissue_name}_final_nodes',
            'path': str(paths['node_output']),
        }
        for tissue_name, paths in tiedie_outputs.items()
    ],
)

display(output_inventory)
